# 12 — Trapped-Ion Backend Comparison

Compare Kuramoto-XY dynamics on superconducting (Heron r2) vs trapped-ion (synthetic) noise models.
All-to-all connectivity eliminates SWAP overhead.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit import transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

from scpn_quantum_control import OMEGA_N_16, QuantumKuramotoSolver, build_knm_paper27
from scpn_quantum_control.hardware.trapped_ion import (
    transpile_for_trapped_ion,
    trapped_ion_noise_model,
)

## 1. Transpilation Comparison

In [ ]:
K = build_knm_paper27(L=4)
solver = QuantumKuramotoSolver(4, K, OMEGA_N_16[:4])
qc = solver.evolve(time=0.3, trotter_steps=2)

qc_ion = transpile_for_trapped_ion(qc)
qc_sc = transpile(qc, basis_gates=["cx", "rz", "sx", "x"], optimization_level=2)

print(f"{'Metric':>20} {'Original':>10} {'Trapped-Ion':>12} {'Supercond.':>12}")
print("-" * 58)
print(f"{'Depth':>20} {qc.depth():>10} {qc_ion.depth():>12} {qc_sc.depth():>12}")
print(f"{'Gates':>20} {qc.size():>10} {qc_ion.size():>12} {qc_sc.size():>12}")
print(
    f"{'2Q gates':>20} {sum(1 for g in qc if g.operation.num_qubits == 2):>10} {sum(1 for g in qc_ion if g.operation.num_qubits == 2):>12} {sum(1 for g in qc_sc if g.operation.num_qubits == 2):>12}"
)

## 2. Noisy Simulation Comparison

In [ ]:
from scpn_quantum_control.hardware.noise_model import heron_r2_noise_model

ion_noise = trapped_ion_noise_model()
sc_noise = heron_r2_noise_model()

# Exact reference
sv = Statevector.from_instruction(qc)
exact_R = (
    float(
        np.abs(
            np.mean(
                [
                    sv.expectation_value("I" * i + "X" + "I" * (3 - i))
                    + 1j * sv.expectation_value("I" * i + "Y" + "I" * (3 - i))
                    for i in range(4)
                ]
            )
        )
    )
    / 4
)

# Noisy simulations
shots = 8192
results = {}
for name, noise, circuit in [("Trapped-Ion", ion_noise, qc_ion), ("Heron r2", sc_noise, qc_sc)]:
    sim = AerSimulator(noise_model=noise)
    qc_m = circuit.copy()
    qc_m.measure_all()
    job = sim.run(qc_m, shots=shots)
    counts = job.result().get_counts()
    total = sum(counts.values())
    z_exps = []
    for q in range(4):
        p0 = sum(v for k, v in counts.items() if k[3 - q] == "0") / total
        z_exps.append(2 * p0 - 1)
    results[name] = z_exps
    print(f"{name}: <Z> per qubit = {[f'{z:.3f}' for z in z_exps]}")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(4)
w = 0.3
ax.bar(x - w, results["Trapped-Ion"], w, label="Trapped-Ion", color="#ff7f0e")
ax.bar(x, results["Heron r2"], w, label="Heron r2", color="#1f77b4")
ax.set_xlabel("Qubit")
ax.set_ylabel("<Z>")
ax.set_title("Noisy Z-expectations: Trapped-Ion vs Superconducting")
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()